In [1]:
import os
import asyncio
from dotenv import load_dotenv
from tavily import TavilyClient

load_dotenv(override=True)

True

In [2]:
tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

In [3]:
# test tim kiem don gian
ket_qua = tavily.search(
    query="xu hướng thương mại điện tử Việt Nam 2026",
    max_results=3
)

for item in ket_qua["results"]:
    print(f" {item['title']}")
    print(f" {item['url']}")
    print(f" {item['content'][:200]}..")
    print()

 Thương mại điện tử 2026: Xu hướng AI-commerce đang nổi lên
 https://dttc.sggp.org.vn/thuong-mai-dien-tu-2026-xu-huong-ai-commerce-dang-noi-len-post130267.html
 Trong bức tranh đó, thương mại điện tử (TMĐT), đặc biệt là TMĐT xuyên biên giới, đang nổi lên như một hướng đi có tính chiến lược...

 Thương mại điện tử năm 2026: Trông đợi cuộc đối đầu gay cấn ...
 https://baodautu.vn/thuong-mai-dien-tu-nam-2026-trong-doi-cuoc-doi-dau-gay-can-giua-2-ong-lon-d536286.html
 Nhìn lại năm 2025, Thứ trưởng Bộ Công thương Nguyễn Sinh Nhật Tn đánh giá, TMĐT Việt Nam tiếp tục khẳng định vai trò trụ cột của kinh tế số với quy mô thị trường ước đạt khoảng 31 tỷ USD, tăng trưởng ..

 Thị trường thương mại điện tử năm 2026 dự báo tăng trưởng cao
 https://congly.vn/thi-truong-thuong-mai-dien-tu-nam-2026-du-bao-tang-truong-cao-508231.html
 Cộng hưởng với dự báo quy mô TMĐT Việt Nam đạt 39 tỷ USD (công bố từ Bộ Công thương) và dự báo tăng trưởng ngành TMĐT với tốc độ trên 20% mỗi..



In [ ]:
from agents import Agent, Runner, trace, function_tool
from agents.model_settings import ModelSettings

@function_tool
def tim_kiem_web(tu_khoa: str) -> str:
    """
    Tìm kiếm thông tin trên internet bằng từ khoá cho trước.
    Trả về tóm tắt các kết quả tìm kiếm liên quan.
    """
    client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    ket_qua = client.search(
        query=tu_khoa,
        max_results=5,          # 5 kết quả/lần tìm = đủ để tổng hợp
        search_depth="basic",   # "basic" nhanh hơn, "advanced" chất lượng hơn
        include_answer=True     # Tavily tự tóm tắt câu trả lời nhanh
    )

    # Format kết quả thành text cho Agent đọc
    output = []

    if ket_qua.get("answer"):
        output.append(f"Tóm tắt nhanh: {ket_qua['answer']}\n")

    for i, item in enumerate(ket_qua["results"], 1):
        output.append(f"[{i}] {item['title']}")
        output.append(f"Nguồn: {item['url']}")
        output.append(f"Nội dung: {item['content'][:400]}")
        output.append("")

    return "\n".join(output)

# Search Agent

In [6]:
# Search Agent
SEARCH_INSTRUCTIONS = """Bạn là trợ lý nghiên cứu thị trường Việt Nam.
Khi được cho một từ khoá tìm kiếm, bạn dùng tool để tìm kiếm thông tin.
Sau đó tổng hợp thành bản tóm tắt ngắn gọn 2-3 đoạn văn, dưới 300 từ.

Quan trọng:
- Tập trung vào thông tin có liên quan đến thị trường Việt Nam khi có thể
- Ghi lại các số liệu, thống kê, tên công ty cụ thể nếu có
- Bỏ qua thông tin chung chung, không có giá trị
- Không cần viết câu hoàn chỉnh — đây là notes cho người tổng hợp báo cáo"""

search_agent = Agent(
    name="Search Agent",
    instructions=SEARCH_INSTRUCTIONS,
    tools=[tim_kiem_web],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required")
)

In [6]:
# test search agent
async def test_search_agent():
    yeu_cau = "xu hướng thương mại điện tử Việt Nam 2026"

    with trace("test-search"):
        result = await Runner.run(search_agent, yeu_cau)
    
    print(result.final_output)

await test_search_agent()

### Tóm tắt xu hướng thương mại điện tử Việt Nam 2026

1. **Tăng trưởng mạnh mẽ**: Thương mại điện tử (TMĐT) Việt Nam dự báo sẽ đạt quy mô 39 tỷ USD vào năm 2026, với tỷ lệ tăng trưởng hằng năm trên 20%. Năm 2025, thị trường ước đạt 31 tỷ USD, cho thấy sự phát triển liên tục của ngành này.

2. **Định hướng chiến lược**: Năm 2026 sẽ chứng kiến sự chuyển mình sang các mô hình kinh doanh tích hợp như AI-commerce (thương mại điện tử thông minh). Đặc biệt, TMĐT xuyên biên giới sẽ trở thành xu hướng chiến lược quan trọng, mang lại cơ hội mở rộng cho các doanh nghiệp.

3. **Thách thức và cơ hội**: Mặc dù số lượng nhà bán hàng trên các nền tảng giảm 5,6% trong năm 2025, doanh thu bình quân trên mỗi nhà bán đã tăng 33%, cho thấy quá trình sàng lọc diễn ra mạnh mẽ. Các doanh nghiệp cần xây dựng thương hiệu và đổi mới để nâng cao hiệu quả cạnh tranh trong bối cảnh thị trường ngày càng phát triển.

### Nguồn tham khảo
- Dự báo và phân tích từ Bộ Công thương, YouNet ECI, và các chuyên gia trong lĩn

# Planner Agent

In [7]:
from pydantic import BaseModel, Field

class KeHoachTimKiem(BaseModel):
    tu_khoa: str = Field(description="Từ khoá cụ thể để tìm kiếm trên Google/Bing")
    ly_do: str = Field(description="Tại sao từ khoá này quan trọng để trả lời câu hỏi nghiên cứu")

class KeHoachNghienCuu(BaseModel):
    cac_tim_kiem: list[KeHoachTimKiem] = Field(description="Danh sách các từ khoá cần tìm kiếm để nghiên cứu toàn diện")

In [9]:
# tao planner Agent
SO_LUONG_TIM_KIEM = 5

PLANNER_INSTRUCTIONS = f"""Bạn là chuyên gia nghiên cứu thị trường Việt Nam.
Khi nhận được một câu hỏi nghiên cứu, bạn lập kế hoạch tìm kiếm thông tin.

Tạo ra đúng {SO_LUONG_TIM_KIEM} từ khoá tìm kiếm để trả lời toàn diện câu hỏi.

Nguyên tắc tạo từ khoá tốt:
- Đa dạng: bao gồm số liệu thị trường, xu hướng, công ty cụ thể, thách thức
- Cụ thể hóa cho Việt Nam khi có thể: thêm "Việt Nam", "TP.HCM", năm hiện tại
- Tránh quá chung chung: "kinh doanh Việt Nam" → "doanh thu ngành F&B Việt Nam 2025-2026"
- Kết hợp tiếng Việt và tiếng Anh nếu chủ đề có nguồn tiếng Anh tốt hơn"""

planner_agent = Agent(
    name="Planner Agent",
    instructions=PLANNER_INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=KeHoachNghienCuu
)


In [10]:
# test 
async def test_planner():
    cau_hoi = "cơ hội và thách thức khi mở chuỗi trà sữa tại Việt Nam năm 2026"

    with trace("test-planner"):
        result = await Runner.run(
            planner_agent,
            cau_hoi
        )
    ke_hoach: KeHoachNghienCuu = result.final_output

    print(f"Kế hoạch tìm kiếm - {len(ke_hoach.cac_tim_kiem)} từ khoá:\n")
    for i, item in enumerate(ke_hoach.cac_tim_kiem, 1):
        print(f"{i}. Từ khoá: {item.tu_khoa}")
        print(f" Lý do: {item.ly_do}\n")

    return ke_hoach

await test_planner()

Kế hoạch tìm kiếm - 5 từ khoá:

1. Từ khoá: xu hướng thị trường trà sữa Việt Nam 2026
 Lý do: Hiểu rõ về thị trường trà sữa, cùng các xu hướng mới nổi để định hướng kinh doanh.

2. Từ khoá: thị trường trà sữa Việt Nam công ty nổi bật 2026
 Lý do: Khám phá các công ty thành công trong ngành trà sữa để rút ra kinh nghiệm và chiến lược.

3. Từ khoá: thách thức khi mở chuỗi trà sữa tại TP.HCM 2026
 Lý do: Xác định những khó khăn có thể gặp phải khi mở chuỗi trà sữa tại một thị trường lớn.

4. Từ khoá: doanh thu ngành trà sữa Việt Nam 2026
 Lý do: Nắm bắt các số liệu về doanh thu và tiềm năng tăng trưởng trong năm 2026.

5. Từ khoá: chiến lược marketing trà sữa tại Việt Nam 2026
 Lý do: Tìm hiểu các chiến lược tiếp thị hiệu quả để thu hút và giữ chân khách hàng.



KeHoachNghienCuu(cac_tim_kiem=[KeHoachTimKiem(tu_khoa='xu hướng thị trường trà sữa Việt Nam 2026', ly_do='Hiểu rõ về thị trường trà sữa, cùng các xu hướng mới nổi để định hướng kinh doanh.'), KeHoachTimKiem(tu_khoa='thị trường trà sữa Việt Nam công ty nổi bật 2026', ly_do='Khám phá các công ty thành công trong ngành trà sữa để rút ra kinh nghiệm và chiến lược.'), KeHoachTimKiem(tu_khoa='thách thức khi mở chuỗi trà sữa tại TP.HCM 2026', ly_do='Xác định những khó khăn có thể gặp phải khi mở chuỗi trà sữa tại một thị trường lớn.'), KeHoachTimKiem(tu_khoa='doanh thu ngành trà sữa Việt Nam 2026', ly_do='Nắm bắt các số liệu về doanh thu và tiềm năng tăng trưởng trong năm 2026.'), KeHoachTimKiem(tu_khoa='chiến lược marketing trà sữa tại Việt Nam 2026', ly_do='Tìm hiểu các chiến lược tiếp thị hiệu quả để thu hút và giữ chân khách hàng.')])

In [10]:
# 3 helper functions
async def lap_ke_hoach(cau_hoi: str) -> KeHoachNghienCuu:

    result = await Runner.run(
        planner_agent,
        cau_hoi
    )
    ke_hoach: KeHoachNghienCuu = result.final_output
    print(f" Sẽ thực hiện {len(ke_hoach.cac_tim_kiem)} tìm kiếm")
    return ke_hoach

async def thuc_hien_mot_tim_kiem(item: KeHoachTimKiem) -> str:
    prompt = f"từ khoá: {item.tu_khoa}\nMục đích: {item.ly_do}"
    result = await Runner.run(search_agent, prompt)
    return result.final_output

async def thuc_hien_tat_ca_tim_kiem(ke_hoach: KeHoachNghienCuu) -> list[str]:
    tasks = [
        asyncio.create_task(thuc_hien_mot_tim_kiem(item))
        for item in ke_hoach.cac_tim_kiem
    ]

    ket_qua = await asyncio.gather(*tasks)
    
    print(f" Hoàn thành {len(ket_qua)} tìm kiếm")
    return list(ket_qua)

In [8]:
# test
async def test_pipeline_tim_kiem():
    import time

    cau_hoi = "Cơ hội và thách thức khi mở chuỗi trà sữa tại Việt Nam năm 2026"
    bat_dau = time.time()

    with trace("pipeline-tim-kiem"):
        ke_hoach = await lap_ke_hoach(cau_hoi)
        ket_qua_tim_kiem = await thuc_hien_tat_ca_tim_kiem(ke_hoach)

    thoi_gian = time.time() - bat_dau
    print(f"Hoàn thành trong {thoi_gian:.1f}s")
    print(f"Tổng kết quả: {sum(len(r) for r in ket_qua_tim_kiem)} ký tự\n")

    print("--Kết quả tìm kiếm #1")
    print(ket_qua_tim_kiem[0][:500])

    return ket_qua_tim_kiem

await test_pipeline_tim_kiem()

 Sẽ thực hiện 5 tìm kiếm
 Hoàn thành 5 tìm kiếm
Hoàn thành trong 19.3s
Tổng kết quả: 4811 ký tự

--Kết quả tìm kiếm #1
- **Tăng trưởng thị trường trà sữa**: Dự báo ngành trà sữa Việt Nam sẽ đạt giá trị 3.43 tỷ USD vào năm 2025 và tăng lên 6.58 tỷ USD vào năm 2034, với tỷ lệ tăng trưởng hàng năm (CAGR) 8.2%. Sự bùng nổ trong tiêu thụ trà sữa phản ánh nhu cầu gia tăng từ người tiêu dùng, đặc biệt ở các đô thị lớn.

- **Cạnh tranh và cơ hội**: Dự báo năm 2026 sẽ chứng kiến sự mở rộng mạnh mẽ của chuỗi cửa hàng trà sữa, phủ kín các khu vực từ phố lớn đến thôn nhỏ. Các công ty lớn như Gong Cha, Koi thương mại sẽ cạnh


['- **Tăng trưởng thị trường trà sữa**: Dự báo ngành trà sữa Việt Nam sẽ đạt giá trị 3.43 tỷ USD vào năm 2025 và tăng lên 6.58 tỷ USD vào năm 2034, với tỷ lệ tăng trưởng hàng năm (CAGR) 8.2%. Sự bùng nổ trong tiêu thụ trà sữa phản ánh nhu cầu gia tăng từ người tiêu dùng, đặc biệt ở các đô thị lớn.\n\n- **Cạnh tranh và cơ hội**: Dự báo năm 2026 sẽ chứng kiến sự mở rộng mạnh mẽ của chuỗi cửa hàng trà sữa, phủ kín các khu vực từ phố lớn đến thôn nhỏ. Các công ty lớn như Gong Cha, Koi thương mại sẽ cạnh tranh mạnh mẽ để chiếm lĩnh thị phần, tạo ra nhiều cơ hội cho những doanh nghiệp nhỏ và mới gia nhập thị trường.\n\n- **Thị trường F&B tổng thể**: Thị trường F&B Việt Nam dự kiến đạt hơn 655 nghìn tỷ đồng vào năm 2024, với mức tăng trưởng 10.92%, chỉ ra rằng trà sữa là một phần không thể thiếu trong bức tranh toàn cảnh này, hỗ trợ cho sự phát triển đa dạng hóa nội dung tiêu dùng.',
 '### Thách thức khi mở chuỗi trà sữa tại TP.HCM 2026\n\n1. **Cạnh tranh khốc liệt**: Thị trường trà sữa tại T

# MarketIQ — AI Research Agent

In [11]:
# schema
from pydantic import BaseModel, Field

class BaoCaoNghienCuu(BaseModel):
    tom_tat_ngan: str = Field(
        description="Tóm tắt ngắn gọn 2-3 câu nêu bật phát hiện quan trọng nhất"
    )
    bao_cao_day_du: str = Field(
        description="""Báo cáo nghiên cứu đầy đủ định dạng Markdown.
        Cấu trúc: Tổng quan → Phân tích chi tiết → Cơ hội → Thách thức → Khuyến nghị.
        Độ dài: 800-1500 từ, có số liệu cụ thể khi có thể."""
    )
    cau_hoi_tiep_theo: list[str] = Field(
        description="3-5 câu hỏi nghiên cứu tiếp theo nên tìm hiểu thêm"
    )

# writer Agent

In [12]:
# writer Agent
WRITER_INSTRUCTIONS = """Bạn là chuyên gia phân tích kinh doanh và nghiên cứu thị trường Việt Nam.

Nhiệm vụ: Dựa trên câu hỏi nghiên cứu và kết quả tìm kiếm được cung cấp,
viết báo cáo phân tích chuyên sâu và có giá trị thực tiễn.

Yêu cầu báo cáo:
- Bắt đầu bằng outline cấu trúc báo cáo, sau đó viết báo cáo đầy đủ
- Ưu tiên số liệu, thống kê, case study cụ thể từ thị trường Việt Nam
- Phân tích từ góc độ người muốn ra quyết định kinh doanh thực tế
- Nêu rõ cơ hội và rủi ro một cách cân bằng
- Định dạng Markdown rõ ràng với các heading, bullet points
- Độ dài 800-1500 từ

Ngôn ngữ: Tiếng Việt, chuyên nghiệp nhưng dễ hiểu — không dùng thuật ngữ chuyên môn
không cần thiết."""

writer_agent = Agent(
    name="Writer Agent",
    instructions=WRITER_INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=BaoCaoNghienCuu
)

# Email Agent 

In [13]:
# Email Agent with Resend
import resend
resend.api_key = os.getenv("RESEND_API_KEY")

@function_tool
def gui_bao_cao_qua_email(tieu_de: str, noi_dung_html: str) -> dict:
    params = {
        "from": "onboarding@resend.dev",
        "to": ["thanhtam.udn@gmail.com"],
        "subject": tieu_de,
        "html": noi_dung_html,
    }

    email = resend.Emails.send(params)
    return {
        "status": "success",
        "email_id": email["id"]
    }

EMAIL_INSTRUCTIONS = """Bạn gửi báo cáo nghiên cứu thị trường qua email.

Quy trình:
1. Chuyển đổi nội dung Markdown sang HTML đẹp, dễ đọc
2. Tạo tiêu đề email mô tả rõ chủ đề nghiên cứu
3. Dùng tool để gửi email

Yêu cầu HTML:
- Font-size 16px, line-height rộng rãi, dễ đọc trên mobile
- Heading màu #2c3e50, body text màu #333
- Highlight số liệu và insight quan trọng bằng bold
- Thêm footer: "Báo cáo được tạo bởi MarketIQ — AI Research Agent"

Tiêu đề email format: "Báo cáo: [chủ đề nghiên cứu]" """

email_agent = Agent(
    name="Email Agent",
    instructions=EMAIL_INSTRUCTIONS,
    tools=[gui_bao_cao_qua_email],
    model="gpt-4o-mini"
)

In [14]:
# helper functions
async def viet_bao_cao(cau_hoi: str, ket_qua_tim_kiem: list[str]) -> BaoCaoNghienCuu:

    noi_dung_dau_vao = f"""Câu hỏi nghiên cứu: {cau_hoi}
Kết quả tìm kiếm:
{chr(10).join(f'[Nguồn {i+1}]{chr(10)}{ket_qua}' for i, ket_qua in enumerate(ket_qua_tim_kiem))}"""

    result = await Runner.run(writer_agent, noi_dung_dau_vao)
    bao_cao: BaoCaoNghienCuu = result.final_output
    return bao_cao

async def gui_email_bao_cao(bao_cao: BaoCaoNghienCuu):
    await Runner.run(email_agent, bao_cao.bao_cao_day_du)
    return bao_cao

In [15]:
# pipeline
from agents import gen_trace_id

async def chay_marketiq(cau_hoi: str):
    trace_id = gen_trace_id()
    print(f"Câu hỏi: {cau_hoi}")
    print(f"Trace ID: {trace_id}\n")

    with trace("marketiq-research", trace_id=trace_id):
        ke_hoach = await lap_ke_hoach(cau_hoi)

        ket_qua_tim_kiem = await thuc_hien_tat_ca_tim_kiem(ke_hoach)

        bao_cao = await viet_bao_cao(cau_hoi, ket_qua_tim_kiem)
        await gui_email_bao_cao(bao_cao)
    
    print(f"Tóm tắt:")
    print(bao_cao.tom_tat_ngan)
    print(f"Gợi ý nghiên cứu tiếp theo:")
    for i, cau in enumerate(bao_cao.cau_hoi_tiep_theo, 1):
        print(f" {i}. {cau}")
    print(f"\n Xem trace đầy đủ:")
    print(f"https://platform.openai.com/logs/trace?trace_id={trace_id}")

    return bao_cao



In [16]:
await chay_marketiq(
    "Mức lương và thị trường tuyển dụng ngành IT tại Việt Nam 2026 — "
    "startup cần chuẩn bị gì để cạnh tranh với tập đoàn lớn?"
)

Câu hỏi: Mức lương và thị trường tuyển dụng ngành IT tại Việt Nam 2026 — startup cần chuẩn bị gì để cạnh tranh với tập đoàn lớn?
Trace ID: trace_fd218dab73e7443190cc54bfee6b467d

 Sẽ thực hiện 5 tìm kiếm
 Hoàn thành 5 tìm kiếm
Tóm tắt:
Mức lương ngành IT tại Việt Nam dự báo sẽ dao động từ 15 triệu đến 53 triệu VNĐ/tháng vào năm 2026, với sự chú ý đặc biệt đến vị trí quản lý AI. Thị trường tuyển dụng dự báo sẽ tăng trưởng mạnh mẽ, với nhu cầu cao cho các vị trí IT chất lượng. Các startup cần chú trọng đến việc xác định thị trường ngách, kỹ năng nhân sự, và công nghệ tiên tiến để cạnh tranh với các tập đoàn lớn.
Gợi ý nghiên cứu tiếp theo:
 1. Các yếu tố nào khác có thể ảnh hưởng đến mức lương ngành IT trong tương lai tại Việt Nam?
 2. Startup cần có những chiến lược gì để thu hút tài năng trong bối cảnh cạnh tranh khốc liệt?
 3. Có những mô hình nào thành công cho startup IT tại Việt Nam trong thời gian qua?
 4. Xu hướng công nghệ nào sẽ là yếu tố quyết định cho sự phát triển của ngành 

BaoCaoNghienCuu(tom_tat_ngan='Mức lương ngành IT tại Việt Nam dự báo sẽ dao động từ 15 triệu đến 53 triệu VNĐ/tháng vào năm 2026, với sự chú ý đặc biệt đến vị trí quản lý AI. Thị trường tuyển dụng dự báo sẽ tăng trưởng mạnh mẽ, với nhu cầu cao cho các vị trí IT chất lượng. Các startup cần chú trọng đến việc xác định thị trường ngách, kỹ năng nhân sự, và công nghệ tiên tiến để cạnh tranh với các tập đoàn lớn.', bao_cao_day_du='# Báo Cáo Nghiên Cứu: Mức Lương và Thị Trường Tuyển Dụng Ngành IT Tại Việt Nam 2026\n\n## Tổng Quan\nNgành IT tại Việt Nam đang phát triển mạnh mẽ, với dự báo về mức lương và nhu cầu tuyển dụng gia tăng đáng kể cho năm 2026. Trong bối cảnh này, các startup đang phải đối mặt với những cơ hội và thách thức lớn, nhất là khi muốn cạnh tranh với các tập đoàn lớn trong ngành.\n\n## Phân Tích Chi Tiết\n### 1. Mức Lương Ngành IT Năm 2026\nTheo các nguồn báo cáo, mức lương trong ngành IT sẽ dao động từ 15 triệu đến 53 triệu VNĐ/tháng:\n- **Vị trí cao cấp**: Quản lý AI có 